<a href="https://colab.research.google.com/github/gordon921212/Baseball-Project/blob/main/baseball.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# install required packages
!pip install pybaseball
!pip install xgboost

In [ ]:
from pybaseball import statcast

df_pitch_by_pitch = statcast(start_dt="2025-03-01", end_dt="2025-11-30")


In [ ]:
df_in_play= df_pitch_by_pitch[df_pitch_by_pitch['description'] == 'hit_into_play']
df_in_play[['launch_speed','launch_angle','events','hit_distance_sc']]

In [ ]:
print(df_in_play['events'].unique())
print("各 events 種類的出現次數：")
print(df_in_play['events'].value_counts())

# 二分類

### Data Pre-processing

In [ ]:
import pandas as pd

print("開始進行精確的標籤編碼...")

# 1. 剔除雜訊：把「捕手妨礙打擊」的無效數據直接刪掉
df_model = df_in_play[df_in_play['events'] != 'catcher_interf'].copy()

# 2. 定義安打的種類
hit_events = ['single', 'double', 'triple', 'home_run']

# 3. 標籤編碼 (Label Encoding)
# 只要是 hit_events 就是 1，其餘 11 種出局或失誤全部歸為 0
df_model['is_hit'] = df_model['events'].apply(lambda x: 1 if x in hit_events else 0)

# 4. 留下我們需要的欄位 (特徵 X 與目標 Y)
df_model = df_model[['launch_speed', 'launch_angle', 'is_hit']]

# 5. 清除雷達當機沒測到初速/仰角的缺失值
df_model = df_model.dropna()

print("========================================")
print(f"✅ 資料清理完成！共保留 {len(df_model)} 筆有效特徵資料。")
print("========================================")

# 檢查一下 0 和 1 的分佈
print(df_model['is_hit'].value_counts(normalize=True) * 100)

### Train-Test Split

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

# =========================
# 1. Prepare X and y
# =========================

# 定義 X 為 launch_speed 和 launch_angle 
# 定義 Y 為 是否打中 (is_hit 在 Data Cleaning 中產生)
X = df_model[['launch_speed', 'launch_angle']]
y = df_model['is_hit']

# 第一次切分：分為 train+validation 和 test
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y,
    test_size=0.2,    # 測試集 佔 20% 
    random_state=42,  # 亂數種子
    stratify=y
)

# 第二次切分：分為 train 和 validation
# Validation 用來提早停止訓練
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full,
    test_size=0.2,
    random_state=42,
    stratify=y_train_full
)


### Train XGBoost binary model

In [ ]:
from xgboost import XGBClassifier
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

# 訓練 XGBoost 模型
xgb_binary_model = XGBClassifier(
    objective='binary:logistic',
    n_estimators=1000,
    learning_rate=0.03,
    max_depth=3,
    min_child_weight=20,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_lambda=1.0,
    reg_alpha=0.0,
    eval_metric='logloss',
    early_stopping_rounds=30,
    tree_method='hist',
    random_state=42,
    n_jobs=-1
)

xgb_binary_model.fit(
    X_train,
    y_train,
    eval_set=[(X_val, y_val)],
    verbose=False
)

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

# Get validation probabilities
y_val_proba = xgb_binary_model.predict_proba(X_val)[:, 1]

# Try many possible thresholds
thresholds = np.arange(0.05, 0.95, 0.01)

best_threshold = 0.5
best_accuracy = 0

for threshold in thresholds:
    y_val_pred_threshold = (y_val_proba >= threshold).astype(int)
    acc = accuracy_score(y_val, y_val_pred_threshold)

    if acc > best_accuracy:
        best_accuracy = acc
        best_threshold = threshold

print("Best threshold:", best_threshold)
print("Best validation accuracy:", best_accuracy)

In [ ]:
# Predicted probability of class 1 = hit
y_test_proba = xgb_binary_model.predict_proba(X_test)[:, 1]

# Prediction using tuned threshold
y_test_pred_threshold = (y_test_proba >= best_threshold).astype(int)

# Accuracy using tuned threshold
tuned_accuracy = accuracy_score(y_test, y_test_pred_threshold)

# ROC-AUC uses probability, not thresholded labels
final_auc = roc_auc_score(y_test, y_test_proba)

print("========================================")
print("XGBoost with Tuned Threshold")
print("========================================")
print(f"Best threshold: {best_threshold:.2f}")
print(f"Tuned Test Accuracy: {tuned_accuracy:.4f}")
print(f"ROC-AUC:             {final_auc:.4f}")

print("\n[Classification Report]")
print(classification_report(
    y_test,
    y_test_pred_threshold,
    target_names=['Out / Non-hit (0)', 'Hit (1)']
))

cm = confusion_matrix(y_test, y_test_pred_threshold)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=['Out / Non-hit (0)', 'Hit (1)']
)

disp.plot(values_format='d')
plt.title("XGBoost Confusion Matrix with Tuned Threshold")
plt.show()

In [ ]:
import pandas as pd

# 1. 確保你使用的是二元分類模型 (0=出局, 1=安打)
# 取得預測機率 (xBA)
xBA_array = xgb_binary_model.predict_proba(X_test)[:, 1]

# 2. 建立一個新的 DataFrame 來進行數據分析
# 把 X_test 複製過來，避免改到原始資料
df_analysis = X_test.copy()

# 加入真實結果與我們算出來的 xBA
df_analysis['Actual_Result'] = y_test
df_analysis['xBA'] = xBA_array

# 3. 算出「運氣值 (Luck Factor)」
# 公式：真實結果 (1或0) - 預期安打率 (xBA)
# 如果結果是 1 (安打)，但 xBA 只有 0.10 -> 運氣值 = +0.90 (超幸運鳥安)
# 如果結果是 0 (出局)，但 xBA 高達 0.95 -> 運氣值 = -0.95 (超衰平飛被接殺)
df_analysis['Luck_Factor'] = df_analysis['Actual_Result'] - df_analysis['xBA']

print("xBA 分析表建置完成！")
print(df_analysis.head())

In [ ]:
# 條件：真實結果是出局(0)，但 xBA 大於 0.90
unlucky_hits = df_analysis[(df_analysis['Actual_Result'] == 0) & (df_analysis['xBA'] > 0.90)]
# 依照運氣值由小到大排序 (越負越衰)
print("🏆 本賽季最衰的 5 次擊球：")
print(unlucky_hits.sort_values(by='Luck_Factor').head(5))

# 三分類

In [ ]:
import pandas as pd

print("開始進行三元分類的標籤編碼...")

# 建立分類函數
def categorize_hit(event):
    if event == 'single':
        return 1  # 短程安打 (一壘安打)
    elif event in ['double', 'triple', 'home_run']:
        return 2  # 長打 (二壘、三壘、全壘打)
    else:
        return 0  # 出局 (其他所有狀況)

# 假設你的原始資料是 df_in_play
df_model = df_in_play[['launch_speed', 'launch_angle', 'events']].copy()

# 套用新的分類邏輯，建立目標變數 y (hit_class)
df_model['hit_class'] = df_model['events'].apply(categorize_hit)

# 移除空值並只留下特徵與目標
df_model = df_model[['launch_speed', 'launch_angle', 'hit_class']].dropna()

# 檢查一下三種類別的分佈比例
print("\n[目標變數 hit_class 的分布比例]")
class_counts = df_model['hit_class'].value_counts()
class_ratio = df_model['hit_class'].value_counts(normalize=True) * 100

print(f"出局 (0):   {class_counts.get(0, 0)} 筆 ({class_ratio.get(0, 0):.2f}%)")
print(f"短程安打 (1): {class_counts.get(1, 0)} 筆 ({class_ratio.get(1, 0):.2f}%)")
print(f"長打 (2):   {class_counts.get(2, 0)} 筆 ({class_ratio.get(2, 0):.2f}%)")

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.utils.class_weight import compute_sample_weight

# 切分特徵 (X) 與新的目標 (y)
X = df_model[['launch_speed', 'launch_angle']]
y = df_model['hit_class']

X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full,
    test_size=0.2,
    random_state=42,
    stratify=y_train_full
)

# =========================
# 3. Handle class imbalance with sample weights
# =========================
sample_weight_train = compute_sample_weight(
    class_weight='balanced',
    y=y_train
)

# =========================
# 4. Train XGBoost multiclass model
# =========================
xgb_multi_model = XGBClassifier(
    objective='multi:softprob',
    num_class=3,
    n_estimators=1000,
    learning_rate=0.03,
    max_depth=4,
    min_child_weight=20,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_lambda=1.0,
    reg_alpha=0.0,
    eval_metric='mlogloss',
    early_stopping_rounds=30,
    tree_method='hist',
    random_state=42,
    n_jobs=-1
)

xgb_multi_model.fit(
    X_train,
    y_train,
    sample_weight=sample_weight_train,
    eval_set=[(X_val, y_val)],
    verbose=False
)

In [ ]:
y_test_pred = xgb_multi_model.predict(X_test)
y_test_proba = xgb_multi_model.predict_proba(X_test)

auc_multi = roc_auc_score(
    y_test,
    y_test_proba,
    multi_class='ovr'
)

print("========================================")
print("XGBoost Three-Class Classification Result")
print("========================================")
print(f"Multiclass ROC-AUC OVR: {auc_multi:.4f}")

target_names = ['出局/非安打 (0)', '短程安打/single (1)', '長打/XBH (2)']

print("\n[Classification Report]")
print(classification_report(y_test, y_test_pred, target_names=target_names))

# =========================
# 6. Confusion matrix
# =========================
cm = confusion_matrix(y_test, y_test_pred)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=target_names
)

disp.plot()
plt.title("XGBoost Three-Class Confusion Matrix")
plt.show()